# DINO World Model on Push-T (LeRobot)

**Paper:** [DINO-WM: World Models on Pre-trained Visual Features enable Zero-shot Planning](https://arxiv.org/abs/2411.04983)

---

In the Ball Arena notebook, we built DINO-WM from scratch on a toy environment. Now we apply it to **Push-T** — a standard robotics benchmark — using real expert demonstrations from the [LeRobot](https://github.com/huggingface/lerobot) dataset.

### Why Push-T is harder:
- **Contact dynamics:** The T-block only moves when the agent pushes it
- **Rotation:** Off-center pushes rotate the T
- **Multi-object scene:** Agent, T-block, and target must all be tracked
- **Expert data:** Real demonstrations with purposeful, coordinated pushing

### What's new vs. Ball Arena:
- **Real dataset** from LeRobot (`lerobot/pusht` — 206 expert episodes)
- **96×96 → 224×224** image upscaling for DINO
- **Pre-encoding** all images with DINO before training (~5x speedup)
- **Action normalization** (actions are target positions in [0, 512])

**Runtime:** ~15 min on T4 GPU

In [ ]:
!pip install -q lerobot einops

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## 1. Load the LeRobot Push-T Dataset

The `lerobot/pusht` dataset contains **206 expert episodes** (25,650 frames at 10 fps) of a circular agent pushing a T-shaped block toward a goal pose.

Each frame provides:
- `observation.image`: (3, 96, 96) RGB image, float32 in [0, 1]
- `observation.state`: (2,) agent position [x, y] in [0, 512]
- `action`: (2,) target agent position [x, y] in [0, 512]

We'll load 40 episodes, truncate to 60 frames each, and resize images to 224×224 for DINO.

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

N_EPISODES = 40
MAX_EP_LEN = 60  # truncate long episodes

print(f"Loading Push-T dataset ({N_EPISODES} episodes)...")
lerobot_ds = LeRobotDataset("lerobot/pusht", episodes=list(range(N_EPISODES)))
print(f"Total frames available: {len(lerobot_ds)}")

In [ ]:
# Extract trajectories: resize images 96x96 → 224x224, group by episode

all_images = []    # will be (total_frames, 3, 224, 224) uint8
all_actions = []   # (total_frames, 2)
all_states = []    # (total_frames, 2)
ep_bounds = []     # (start_idx, end_idx) per episode

prev_ep = None
ep_frame_count = 0
ep_start = 0

for i in tqdm(range(len(lerobot_ds)), desc="Extracting & resizing"):
    sample = lerobot_ds[i]
    ep = sample["episode_index"].item()

    # Track episode boundaries
    if ep != prev_ep:
        if prev_ep is not None:
            ep_bounds.append((ep_start, len(all_images)))
        ep_start = len(all_images)
        ep_frame_count = 0
        prev_ep = ep

    ep_frame_count += 1
    if ep_frame_count > MAX_EP_LEN:
        continue

    # Image: (3, 96, 96) float → resize to (3, 224, 224) → store as uint8
    img = sample["observation.image"]  # (3, 96, 96) float32 [0, 1]
    img_224 = F.interpolate(img.unsqueeze(0), size=(224, 224),
                            mode='bilinear', align_corners=False).squeeze(0)
    all_images.append((img_224 * 255).byte())  # uint8 saves memory
    all_actions.append(sample["action"])
    all_states.append(sample["observation.state"])

# Don't forget last episode
ep_bounds.append((ep_start, len(all_images)))

# Stack into tensors
images = torch.stack(all_images)    # (total, 3, 224, 224) uint8
actions = torch.stack(all_actions)  # (total, 2)
states = torch.stack(all_states)    # (total, 2)

print(f"\nExtracted {len(images)} frames from {len(ep_bounds)} episodes")
print(f"Images: {images.shape} ({images.element_size() * images.nelement() / 1e9:.2f} GB)")
print(f"Actions range: [{actions.min():.1f}, {actions.max():.1f}]")
print(f"States range:  [{states.min():.1f}, {states.max():.1f}]")
print(f"Episode lengths: {[e-s for s,e in ep_bounds[:5]]}... (first 5)")

In [ ]:
# Normalize actions and states (zero-mean, unit-variance)
# This is important — raw values are in [0, 512]

act_mean, act_std = actions.mean(0), actions.std(0)
state_mean, state_std = states.mean(0), states.std(0)

actions_norm = (actions - act_mean) / (act_std + 1e-8)
states_norm = (states - state_mean) / (state_std + 1e-8)

print(f"Action  mean={act_mean.numpy().round(1)}, std={act_std.numpy().round(1)}")
print(f"State   mean={state_mean.numpy().round(1)}, std={state_std.numpy().round(1)}")
print(f"After normalization — actions: [{actions_norm.min():.2f}, {actions_norm.max():.2f}]")

In [ ]:
# Visualize expert demonstrations
fig, axes = plt.subplots(3, 8, figsize=(20, 7.5))
for row in range(3):
    start, end = ep_bounds[row * 10]
    ep_len = end - start
    for col in range(8):
        t = start + min(col * (ep_len // 8), ep_len - 1)
        img = images[t].permute(1, 2, 0).numpy()  # uint8 HWC
        axes[row, col].imshow(img)
        axes[row, col].set_title(f'f={col * (ep_len // 8)}', fontsize=8)
        axes[row, col].axis('off')
fig.suptitle('Expert Push-T Demonstrations from LeRobot (every ~8th frame)', fontsize=12)
plt.tight_layout()
plt.show()
print("These are EXPERT demonstrations — the agent deliberately pushes the T to the target.")

---
## 2. DINO Encoder + Pre-encoding

Same frozen DINOv2 encoder. We pre-encode all images upfront since DINO is frozen — its output is deterministic, so no reason to recompute every epoch.

In [ ]:
class DINOEncoder(nn.Module):
    """Frozen DINOv2 ViT-S/14: image → 256 patch embeddings of dim 384."""
    def __init__(self):
        super().__init__()
        self.dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
        self.dino.eval()
        for p in self.dino.parameters():
            p.requires_grad = False
        self.embed_dim = 384
        self.num_patches = 256

    @torch.no_grad()
    def forward(self, x):
        return self.dino.forward_features(x)['x_norm_patchtokens']

dino_encoder = DINOEncoder().to(device)
print(f"DINO encoder: {sum(p.numel() for p in dino_encoder.parameters())/1e6:.1f}M params (frozen)")

In [ ]:
# Pre-encode all images with DINO (batch processing)
print(f"Pre-encoding {len(images)} images with DINO...")
features_list = []
encode_bs = 32

for i in tqdm(range(0, len(images), encode_bs)):
    batch = images[i:i+encode_bs].float().div(255.0).to(device)  # uint8 → float [0,1]
    feats = dino_encoder(batch)  # (B, 256, 384)
    features_list.append(feats.cpu())

dino_features = torch.cat(features_list, dim=0)  # (total, 256, 384)
print(f"Features: {dino_features.shape} ({dino_features.element_size() * dino_features.nelement() / 1e9:.2f} GB)")

In [ ]:
# Visualize DINO features on Push-T
from sklearn.decomposition import PCA

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
ep_start, ep_end = ep_bounds[0]
frame_indices = np.linspace(ep_start, ep_end - 1, 5, dtype=int)

sel_feats = [dino_features[fi].numpy() for fi in frame_indices]
pca = PCA(n_components=3).fit(np.concatenate(sel_feats))

for i, fi in enumerate(frame_indices):
    axes[0, i].imshow(images[fi].permute(1, 2, 0).numpy())
    axes[0, i].set_title(f'Frame {fi - ep_start}', fontsize=9); axes[0, i].axis('off')

    fp = pca.transform(sel_feats[i]).reshape(16, 16, 3)
    fp = (fp - fp.min()) / (fp.max() - fp.min() + 1e-8)
    axes[1, i].imshow(fp); axes[1, i].set_title('DINO PCA', fontsize=9); axes[1, i].axis('off')

fig.suptitle('DINO features on Push-T expert demonstrations', fontsize=12)
plt.tight_layout(); plt.show()

---
## 3. World Model Architecture

Same components as Ball Arena. Training uses pre-computed DINO features; inference encodes on the fly.

In [ ]:
# ── Encoders ──────────────────────────────────────────────────────────

class ActionEncoder(nn.Module):
    def __init__(self, dim=2, embed_dim=384, n_tok=4):
        super().__init__()
        self.n, self.d = n_tok, embed_dim
        self.proj = nn.Sequential(nn.Linear(dim, 128), nn.GELU(), nn.Linear(128, n_tok * embed_dim))
    def forward(self, x): return self.proj(x).reshape(-1, self.n, self.d)

class ProprioEncoder(nn.Module):
    def __init__(self, dim=2, embed_dim=384, n_tok=4):
        super().__init__()
        self.n, self.d = n_tok, embed_dim
        self.proj = nn.Sequential(nn.Linear(dim, 128), nn.GELU(), nn.Linear(128, n_tok * embed_dim))
    def forward(self, x): return self.proj(x).reshape(-1, self.n, self.d)

# ── Transformer Predictor ─────────────────────────────────────────────

class TransformerPredictor(nn.Module):
    def __init__(self, embed_dim=384, depth=4, num_heads=8, mlp_ratio=4,
                 num_patches=256, num_extra=8, num_hist=3):
        super().__init__()
        self.num_patches = num_patches
        self.tps = num_patches + num_extra  # tokens per step
        self.spatial_pos = nn.Parameter(torch.randn(1, self.tps, embed_dim) * 0.02)
        self.temporal_pos = nn.Parameter(torch.randn(1, num_hist + 1, embed_dim) * 0.02)
        mlp_dim = embed_dim * mlp_ratio
        layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads,
            dim_feedforward=mlp_dim, dropout=0.1, activation='gelu',
            batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(nn.Linear(embed_dim, mlp_dim), nn.GELU(), nn.Linear(mlp_dim, embed_dim))

    def _mask(self, S, dev):
        ids = torch.arange(S, device=dev) // self.tps
        m = torch.zeros(S, S, device=dev)
        m.masked_fill_(ids.unsqueeze(0) > ids.unsqueeze(1), float('-inf'))
        return m

    def forward(self, z):
        B, S, D = z.shape
        n = S // self.tps
        z = z + self.spatial_pos.repeat(1, n, 1)
        z = z + self.temporal_pos[:, :n].repeat_interleave(self.tps, dim=1)
        z = self.norm(self.transformer(z, mask=self._mask(S, z.device)))
        s = (n - 1) * self.tps
        return self.head(z[:, s:s + self.num_patches])

# ── VQ-VAE Decoder ────────────────────────────────────────────────────

class VectorQuantizer(nn.Module):
    def __init__(self, K=512, D=384, beta=0.25):
        super().__init__()
        self.beta = beta
        self.cb = nn.Embedding(K, D)
        self.cb.weight.data.uniform_(-1/K, 1/K)
    def forward(self, z):
        B,H,W,D = z.shape; flat = z.reshape(-1, D)
        d = flat.pow(2).sum(1,True) + self.cb.weight.pow(2).sum(1) - 2*flat@self.cb.weight.t()
        zq = self.cb(d.argmin(1)).reshape(B,H,W,D)
        loss = F.mse_loss(zq.detach(),z) + self.beta*F.mse_loss(zq,z.detach())
        return z + (zq-z).detach(), loss

class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.b = nn.Sequential(nn.GroupNorm(8,c),nn.GELU(),nn.Conv2d(c,c,3,padding=1),
                               nn.GroupNorm(8,c),nn.GELU(),nn.Conv2d(c,c,3,padding=1))
    def forward(self, x): return x + self.b(x)

class VQVAEDecoder(nn.Module):
    def __init__(self, D=384, K=512):
        super().__init__()
        self.D = D; self.pq = nn.Linear(D,D); self.vq = VectorQuantizer(K,D)
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(D,256,4,2,1), ResBlock(256),
            nn.ConvTranspose2d(256,128,4,2,1), ResBlock(128),
            nn.ConvTranspose2d(128,64,4,2,1), ResBlock(64),
            nn.Upsample(size=(224,224),mode='bilinear',align_corners=False),
            nn.Conv2d(64,32,3,padding=1),nn.GELU(),nn.Conv2d(32,3,3,padding=1),nn.Sigmoid())
    def forward(self, z):
        B = z.shape[0]
        zq, vql = self.vq(self.pq(z).reshape(B,16,16,self.D))
        return self.dec(zq.permute(0,3,1,2)), vql

print("Architecture defined.")

In [ ]:
class DinoWorldModel(nn.Module):
    def __init__(self, act_dim=2, prop_dim=2, D=384, NP=256,
                 at=4, pt=4, nh=3, depth=4, heads=8):
        super().__init__()
        self.nh, self.NP = nh, NP
        self.dino = DINOEncoder()
        self.act_enc = ActionEncoder(act_dim, D, at)
        self.prop_enc = ProprioEncoder(prop_dim, D, pt)
        self.predictor = TransformerPredictor(D, depth, heads, num_patches=NP,
                                              num_extra=at+pt, num_hist=nh)
        self.decoder = VQVAEDecoder(D)

    def _tok(self, zv, a, p):
        return torch.cat([zv, self.act_enc(a), self.prop_enc(p)], dim=1)

    def forward_preencoded(self, feats, tgt_img, acts, props):
        """Train with pre-computed DINO features.
        feats: (B,T,256,384), tgt_img: (B,3,224,224), acts/props: (B,T,2)"""
        hist = torch.cat([self._tok(feats[:,t], acts[:,t], props[:,t])
                          for t in range(self.nh)], dim=1)
        zp = self.predictor(hist)
        zt = feats[:, self.nh]

        z_loss = F.mse_loss(zp, zt)
        ip, vqp = self.decoder(zp); rl = F.mse_loss(ip, tgt_img)
        ig, vqg = self.decoder(zt.detach()); rg = F.mse_loss(ig, tgt_img)
        total = z_loss + rl + 0.25*vqp + rg + 0.25*vqg
        return {'total_loss': total, 'z_loss': z_loss.item(),
                'recon_loss': rl.item(), 'recon_gt': rg.item(), 'vq_loss': (vqp+vqg).item()}

    @torch.no_grad()
    def predict_from_images(self, vis, acts, props):
        """Inference: vis (1,nh,3,224,224), acts/props (1,nh,2) → img (1,3,224,224)"""
        toks = []
        for t in range(self.nh):
            zv = self.dino(vis[:,t])
            toks.append(self._tok(zv, acts[:,t], props[:,t]))
        zp = self.predictor(torch.cat(toks, dim=1))
        ip, _ = self.decoder(zp)
        return ip, zp

model = DinoWorldModel(act_dim=2, prop_dim=2, nh=3).to(device)
tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
tot = sum(p.numel() for p in model.parameters())
print(f"Trainable: {tr/1e6:.1f}M / Total: {tot/1e6:.1f}M")

---
## 4. Dataset & Training

In [ ]:
class PushTSliceDataset(torch.utils.data.Dataset):
    """Slices episodes into (num_hist+1) windows, respecting episode boundaries."""

    def __init__(self, features, images, actions, states, ep_bounds, num_hist=3):
        self.w = num_hist + 1
        self.features = features
        self.images = images
        self.actions = actions
        self.states = states
        # Build valid window starts (never cross episode boundaries)
        self.starts = []
        for s, e in ep_bounds:
            for t in range(s, e - self.w + 1):
                self.starts.append(t)

    def __len__(self): return len(self.starts)

    def __getitem__(self, idx):
        t = self.starts[idx]
        s = slice(t, t + self.w)
        feats = self.features[s]                         # (W, 256, 384)
        tgt = self.images[t + self.w - 1].float() / 255  # (3, 224, 224)
        return feats, tgt, self.actions[s], self.states[s]


dataset = PushTSliceDataset(dino_features, images, actions_norm, states_norm,
                            ep_bounds, num_hist=3)
loader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True,
                                     num_workers=0, pin_memory=True)
print(f"Dataset: {len(dataset)} windows, {len(loader)} batches/epoch")

In [ ]:
optimizer = torch.optim.AdamW([
    {'params': model.predictor.parameters(), 'lr': 5e-4},
    {'params': model.decoder.parameters(), 'lr': 3e-4},
    {'params': model.act_enc.parameters(), 'lr': 5e-4},
    {'params': model.prop_enc.parameters(), 'lr': 5e-4},
], weight_decay=1e-4)

EPOCHS = 30
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
hist = {k: [] for k in ['total_loss','z_loss','recon_loss','recon_gt','vq_loss']}

model.train(); model.dino.eval()

for epoch in range(EPOCHS):
    ep_l = {k: [] for k in hist}
    pbar = tqdm(loader, desc=f'Epoch {epoch+1}/{EPOCHS}', leave=False)
    for feats, tgt, acts, props in pbar:
        ld = model.forward_preencoded(feats.to(device), tgt.to(device),
                                      acts.to(device), props.to(device))
        optimizer.zero_grad(); ld['total_loss'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        for k in hist:
            v = ld[k]; ep_l[k].append(v.item() if isinstance(v, torch.Tensor) else v)
        pbar.set_postfix(z=f"{ld['z_loss']:.4f}", r=f"{ld['recon_loss']:.4f}")
    sched.step()
    for k in hist: hist[k].append(np.mean(ep_l[k]))
    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d} | Total: {hist['total_loss'][-1]:.4f} | "
              f"Z: {hist['z_loss'][-1]:.4f} | Recon: {hist['recon_loss'][-1]:.4f} | "
              f"Recon-GT: {hist['recon_gt'][-1]:.4f}")
print("Done!")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, k, t, c in [
    (axes[0], 'z_loss', 'Embedding Prediction', '#5B8CB8'),
    (axes[1], 'recon_loss', 'Reconstruction (pred)', '#D97757'),
    (axes[2], 'recon_gt', 'Reconstruction (GT)', '#7DA488')]:
    ax.plot(hist[k], color=c, lw=2); ax.set_title(t); ax.set_xlabel('Epoch'); ax.grid(alpha=0.3)
fig.suptitle('Push-T Training (LeRobot expert data)', fontweight='bold')
plt.tight_layout(); plt.show()

---
## 5. Next-Frame Prediction

3 history frames → predict the 4th. On expert Push-T data, this means predicting purposeful pushing motions.

In [ ]:
model.eval()

n_show = 5
fig, axes = plt.subplots(n_show, 5, figsize=(15, 3 * n_show))

for row in range(n_show):
    # Pick a window from different episodes
    ep_idx = row * 8
    s, e = ep_bounds[ep_idx]
    t0 = s + min(10, (e - s) // 3)  # start partway into episode

    vis = images[t0:t0+3].float().div(255.0).unsqueeze(0).to(device)
    act = actions_norm[t0:t0+3].unsqueeze(0).to(device)
    prop = states_norm[t0:t0+3].unsqueeze(0).to(device)

    pred, _ = model.predict_from_images(vis, act, prop)

    for c in range(3):
        axes[row, c].imshow(images[t0+c].permute(1,2,0).numpy())
        axes[row, c].set_title(f'History {c}', fontsize=9); axes[row, c].axis('off')

    axes[row, 3].imshow(pred[0].permute(1,2,0).cpu().numpy().clip(0,1))
    axes[row, 3].set_title('Predicted', fontsize=9, color='#D97757', fontweight='bold')
    axes[row, 3].axis('off')

    axes[row, 4].imshow(images[t0+3].permute(1,2,0).numpy())
    axes[row, 4].set_title('Actual', fontsize=9, color='#7DA488', fontweight='bold')
    axes[row, 4].axis('off')

fig.suptitle('Push-T Next-Frame Prediction on Expert Data', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 6. Multi-Step Rollout

In [ ]:
@torch.no_grad()
def rollout(model, init_vis, init_act, init_prop, fut_act, n_steps):
    model.eval()
    vb = list(init_vis[0].cpu())
    ab = list(init_act[0].cpu())
    pb = list(init_prop[0].cpu())
    preds = []
    for s in range(n_steps):
        v = torch.stack(vb[-3:]).unsqueeze(0).to(device)
        a = torch.stack(ab[-3:]).unsqueeze(0).to(device)
        p = torch.stack(pb[-3:]).unsqueeze(0).to(device)
        ip, _ = model.predict_from_images(v, a, p)
        preds.append(ip[0].cpu())
        vb.append(ip[0].cpu()); ab.append(fut_act[0,s].cpu()); pb.append(pb[-1])
    return preds

# Pick a test episode
ep_idx = 35
s0, e0 = ep_bounds[ep_idx]
t0 = s0 + 5
n_fut = min(8, e0 - t0 - 3)

iv = images[t0:t0+3].float().div(255.0).unsqueeze(0)
ia = actions_norm[t0:t0+3].unsqueeze(0)
ip = states_norm[t0:t0+3].unsqueeze(0)
fa = actions_norm[t0+3:t0+3+n_fut].unsqueeze(0)

pf = rollout(model, iv, ia, ip, fa, n_fut)

fig, axes = plt.subplots(2, n_fut, figsize=(2.3*n_fut, 5))
for t in range(n_fut):
    axes[0,t].imshow(pf[t].permute(1,2,0).numpy().clip(0,1))
    axes[0,t].set_title(f'Pred +{t+1}', fontsize=8, color='#D97757'); axes[0,t].axis('off')
    at = t0+3+t
    if at < e0:
        axes[1,t].imshow(images[at].permute(1,2,0).numpy())
    axes[1,t].set_title(f'Actual +{t+1}', fontsize=8, color='#7DA488'); axes[1,t].axis('off')
fig.suptitle(f'Autoregressive Rollout on Expert Push-T', fontweight='bold')
plt.tight_layout(); plt.show()

---
## 7. Zero-Shot Planning with CEM

Pick a **start** frame and a **goal** frame from a held-out episode. CEM plans actions to reach the goal purely in DINO embedding space — the model was never trained on goal-reaching.

In [ ]:
class CEMPlanner:
    def __init__(self, model, horizon=6, n_samples=200, topk=20, opt_steps=15):
        self.model = model; self.H = horizon
        self.ns = n_samples; self.topk = topk; self.opt = opt_steps

    @torch.no_grad()
    def plan(self, init_vis, init_act, init_prop, goal_img):
        self.model.eval()
        z_goal = self.model.dino(goal_img.to(device))
        # Pre-encode history
        zt = []
        for t in range(3):
            zv = self.model.dino(init_vis[:,t].to(device))
            zt.append(self.model._tok(zv, init_act[:,t].to(device), init_prop[:,t].to(device)))

        mean = torch.zeros(self.H, 2, device=device)
        std = torch.ones(self.H, 2, device=device) * 1.5
        best_a, best_c = None, float('inf')

        for _ in range(self.opt):
            acts = (mean + std * torch.randn(self.ns, self.H, 2, device=device)).clamp(-3, 3)
            costs = []
            for i in range(self.ns):
                zf = self._sim(zt, init_prop, acts[i])
                costs.append(F.mse_loss(zf, z_goal).item())
            costs = torch.tensor(costs, device=device)
            ti = costs.argsort()[:self.topk]
            if costs[ti[0]] < best_c:
                best_c = costs[ti[0]].item(); best_a = acts[ti[0]].clone()
            mean = acts[ti].mean(0); std = acts[ti].std(0).clamp(min=0.1)
        return best_a.cpu(), best_c

    def _sim(self, init_zt, init_prop, aseq):
        az = list(init_zt)
        for s in range(len(aseq)):
            zp = self.model.predictor(torch.cat(az[-3:], dim=1))
            at = self.model.act_enc(aseq[s:s+1].unsqueeze(0))
            pt = self.model.prop_enc(init_prop[:,-1].to(device))
            az.append(torch.cat([zp, at, pt], dim=1))
        return az[-1][:, :self.model.NP]

print("CEM Planner ready.")

In [ ]:
# Planning demo: start from early in episode, goal = END of episode (T aligned with target)
test_ep = 38
ts, te = ep_bounds[test_ep]
t_start = ts + 2                # early in the episode (T not yet aligned)
t_goal = te - 1                 # END of episode (T aligned with target!)

print(f"Episode {test_ep}: {te - ts} frames")
print(f"Start: frame {t_start - ts} (early — T far from target)")
print(f"Goal:  frame {t_goal - ts} (final — T aligned with target)")
print(f"Gap:   {t_goal - t_start} frames apart")

start_vis = images[t_start:t_start+3].float().div(255.0).unsqueeze(0)
start_act = actions_norm[t_start:t_start+3].unsqueeze(0)
start_prop = states_norm[t_start:t_start+3].unsqueeze(0)
goal_frame = images[t_goal].float().div(255.0).unsqueeze(0)

# Show start and goal
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
for i in range(3):
    axes[i].imshow(images[t_start+i].permute(1,2,0).numpy())
    axes[i].set_title(f'History {i} (early)', fontsize=9); axes[i].axis('off')
axes[3].imshow(images[(ts + te) // 2].permute(1,2,0).numpy())
axes[3].set_title('Mid-episode', fontsize=9, color='gray'); axes[3].axis('off')
axes[4].imshow(images[t_goal].permute(1,2,0).numpy())
axes[4].set_title('GOAL (T aligned!)', color='#7DA488', fontweight='bold', fontsize=10)
axes[4].axis('off')
fig.suptitle(f'Planning task: push T from start → aligned with target (episode {test_ep})', fontsize=12)
plt.tight_layout(); plt.show()

# Plan with longer horizon since goal is further away
planner = CEMPlanner(model, horizon=8, n_samples=300, topk=30, opt_steps=20)
print("Planning to reach the aligned T configuration...")
planned, cost = planner.plan(start_vis, start_act, start_prop, goal_frame)
print(f"Done! Cost: {cost:.6f}")

In [ ]:
# Visualize: model's imagined trajectory toward aligned T vs actual expert trajectory
n_plan = 8  # matches horizon=8
pred_frames = rollout(model, start_vis, start_act, start_prop,
                      planned.unsqueeze(0), n_steps=n_plan)

fig, axes = plt.subplots(2, n_plan + 2, figsize=(22, 5))
# Row 0: imagined trajectory (planned by CEM)
axes[0,0].imshow(images[t_start+2].permute(1,2,0).numpy())
axes[0,0].set_title('Start', fontsize=9); axes[0,0].axis('off')
for i, pf in enumerate(pred_frames):
    axes[0,i+1].imshow(pf.permute(1,2,0).numpy().clip(0,1))
    axes[0,i+1].set_title(f'Imagined +{i+1}', fontsize=8, color='#D97757'); axes[0,i+1].axis('off')
axes[0,n_plan+1].imshow(images[t_goal].permute(1,2,0).numpy())
axes[0,n_plan+1].set_title('GOAL\n(T aligned)', fontsize=9, color='#7DA488', fontweight='bold')
axes[0,n_plan+1].axis('off')

# Row 1: actual expert trajectory from start to end of episode
actual_frames = np.linspace(t_start+2, t_goal, n_plan + 2, dtype=int)
for i, fi in enumerate(actual_frames):
    axes[1,i].imshow(images[fi].permute(1,2,0).numpy())
    if i == 0:
        lbl = 'Start'
    elif i == len(actual_frames) - 1:
        lbl = 'GOAL\n(T aligned)'
    else:
        lbl = f'Expert'
    c = '#7DA488' if i == len(actual_frames) - 1 else 'black'
    axes[1,i].set_title(lbl, fontsize=8, color=c); axes[1,i].axis('off')

axes[0,0].set_ylabel('CEM Planned\n(imagined)', fontsize=10)
axes[1,0].set_ylabel('Expert\n(actual)', fontsize=10)
fig.suptitle('Zero-Shot Planning: Push T to Align with Target', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print("Top row: CEM plans actions in DINO embedding space to reach the aligned goal.")
print("Bottom row: what the expert actually did in the dataset.")
print("The world model was NEVER trained on goal-reaching — this is zero-shot!")

---
## 8. Embedding Space Analysis

In [ ]:
from sklearn.manifold import TSNE

# Collect embeddings from multiple episodes at different time points
sample_indices = []
time_fracs = []  # 0=start, 1=end of episode
ep_ids = []

for ei in range(0, min(30, len(ep_bounds))):
    s, e = ep_bounds[ei]
    elen = e - s
    for frac in np.linspace(0, 1, 6):
        fi = s + int(frac * (elen - 1))
        sample_indices.append(fi)
        time_fracs.append(frac)
        ep_ids.append(ei)

# Get mean-pooled DINO features
embs = dino_features[sample_indices].mean(dim=1).numpy()  # (N, 384)
time_fracs = np.array(time_fracs)

tsne = TSNE(n_components=2, random_state=42, perplexity=20)
e2d = tsne.fit_transform(embs)

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(e2d[:,0], e2d[:,1], c=time_fracs, cmap='coolwarm',
                s=40, edgecolors='white', linewidths=0.3)
plt.colorbar(sc, ax=ax, label='Time in episode (0=start, 1=end)')
ax.set_title('DINO Embedding Space (t-SNE)\n'
             'Blue = early (T far from goal) → Red = late (T near goal)', fontsize=11)
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
plt.tight_layout(); plt.show()

print("Early frames (T far from goal) cluster separately from late frames (T at goal).")
print("DINO captures the progression of the task — this is what makes planning work!")

---
## Summary

We applied **DINO-WM** to the standard **Push-T** benchmark using **expert demonstrations from LeRobot**.

### Key results:
| What | Details |
|------|--------|
| **Data** | 40 expert episodes from `lerobot/pusht` (LeRobot / HuggingFace) |
| **Pre-encoding** | All 96×96 images resized to 224×224 and encoded with frozen DINO upfront |
| **Training** | ~5 min with pre-encoded features (no DINO in the training loop) |
| **Prediction** | Model predicts T-block movement, rotation, and agent motion |
| **Planning** | CEM plans in DINO embedding space to reach goal configurations |

### vs. Ball Arena:
- **Real benchmark** with contact dynamics, rotation, and expert demonstrations
- **Real dataset** from a standard robotics research platform
- **Harder dynamics** — the T only moves on contact, and rotation matters

### Going further:
- Install `gym-pusht` (`pip install 'lerobot[pusht]'`) to execute CEM plans in the real simulator
- Train on all 206 episodes for better dynamics
- Try gradient-based planning (backprop through the world model)
- Scale to real robot tasks (the paper shows results on rope + granular manipulation)

### References
- [DINO-WM Paper](https://arxiv.org/abs/2411.04983) | [Code](https://github.com/gaoyuezhou/dino_wm)
- [LeRobot](https://github.com/huggingface/lerobot) | [Push-T dataset](https://huggingface.co/datasets/lerobot/pusht)
- [DINOv2](https://github.com/facebookresearch/dinov2)